In [1]:
import pandas as pd
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_colwidth', 50)

df = pd.read_csv('../data/processed(k=5)/base_human_head_k5.csv')
print(f"Shape: {df.shape[0]} righe × {df.shape[1]} colonne")
print(f"Istanze uniche: {df['instance_id'].nunique()}")
print(f"Celle vuote: {df.isna().sum().sum() + (df == '').sum().sum()}")

Shape: 878 righe × 134 colonne
Istanze uniche: 131
Celle vuote: 6688


In [2]:
# Mostra come sono organizzate le colonne
cols = df.columns.tolist()
print("=== IDENTIFICATORI (4) ===")
for c in cols[:4]:
    print(f"  {c}")

print(f"\n=== METRICHE BASE (13 metriche × 5 attempt = 65 colonne) ===")
base_cols = [c for c in cols if '_base_' in c]
# Raggruppa per metrica
from collections import OrderedDict
metrics_base = OrderedDict()
for c in base_cols:
    metric = c.replace('_base_Attempt1','').replace('_base_Attempt2','').replace('_base_Attempt3','').replace('_base_Attempt4','').replace('_base_Attempt5','')
    if metric not in metrics_base:
        metrics_base[metric] = []
    metrics_base[metric].append(c)

for metric, attempts in metrics_base.items():
    print(f"  {metric}: [{', '.join([a.split('Attempt')[1] for a in attempts])}]")

print(f"\n=== METRICHE HUMAN_HEAD (13 metriche × 5 attempt = 65 colonne) ===")
hh_cols = [c for c in cols if '_human_head_' in c]
print(f"  (stessa struttura di base, {len(hh_cols)} colonne)")

=== IDENTIFICATORI (4) ===
  instance_id
  test_name
  repo
  path

=== METRICHE BASE (13 metriche × 5 attempt = 65 colonne) ===
  cpu_energy_joules: [1, 2, 3, 4, 5]
  gpu_energy_joules: [1, 2, 3, 4, 5]
  total_energy_joules: [1, 2, 3, 4, 5]
  system_energy_joules: [1, 2, 3, 4, 5]
  power_watts: [1, 2, 3, 4, 5]
  carbon_grams: [1, 2, 3, 4, 5]
  carbon_grams_system: [1, 2, 3, 4, 5]
  duration_seconds: [1, 2, 3, 4, 5]
  cpu_usage_mean_percent: [1, 2, 3, 4, 5]
  cpu_usage_peak_percent: [1, 2, 3, 4, 5]
  ram_usage_mean_mb: [1, 2, 3, 4, 5]
  ram_usage_peak_mb: [1, 2, 3, 4, 5]
  gpu_temperature_mean_celsius: [1, 2, 3, 4, 5]

=== METRICHE HUMAN_HEAD (13 metriche × 5 attempt = 65 colonne) ===
  (stessa struttura di base, 65 colonne)


In [5]:
# Cella 3 - senza .style
id_cols = ['instance_id', 'test_name', 'repo']
duration_cols = [c for c in df.columns if 'duration_seconds' in c]

df[id_cols + duration_cols].head(5).round(2)

,instance_id,test_name,repo,duration_seconds_base_Attempt1,duration_seconds_base_Attempt2,duration_seconds_base_Attempt3,duration_seconds_base_Attempt4,duration_seconds_base_Attempt5,duration_seconds_human_head_Attempt1,duration_seconds_human_head_Attempt2,duration_seconds_human_head_Attempt3,duration_seconds_human_head_Attempt4,duration_seconds_human_head_Attempt5
0,astropy__astropy-16058,astropy/timeseries/periodograms/lombscargle/te...,astropy/astropy,1.6,1.2,1.30,1.2,1.3,1.7,1.3,1.2,1.3,1.3
1,astropy__astropy-15948,astropy/timeseries/periodograms/lombscargle/im...,astropy/astropy,1.6,1.2,1.11,1.2,1.2,1.7,1.2,1.2,1.2,1.2
2,astropy__astropy-16065,astropy/timeseries/periodograms/lombscargle/te...,astropy/astropy,1.7,1.2,1.30,1.3,1.2,1.7,1.3,1.3,1.3,1.2
3,astropy__astropy-16065,astropy/units/tests/test_quantity_decorator.py...,astropy/astropy,1.3,1.2,1.30,1.2,1.2,1.2,1.3,1.2,1.2,1.3
4,astropy__astropy-15710,astropy/utils/tests/test_data.py::test_update_...,astropy/astropy,1.9,1.2,1.20,1.2,1.2,1.8,1.3,1.2,1.2,1.3


In [8]:
# Prendi la prima riga e mostra tutte le 134 colonne in verticale
row = df.iloc[0]
print(f"Instance: {row['instance_id']}")
print(f"Test: {row['test_name']}")
print(f"Repo: {row['repo']}\n")

for config in ['base', 'human_head']:
    print(f"{'='*60}")
    print(f"  {config.upper()}")
    print(f"{'='*60}")
    for metric in ['cpu_energy_joules', 'gpu_energy_joules', 'total_energy_joules', 
                    'system_energy_joules', 'power_watts', 'carbon_grams', 'carbon_grams_system',
                    'duration_seconds', 'cpu_usage_mean_percent', 'cpu_usage_peak_percent',
                    'ram_usage_mean_mb', 'ram_usage_peak_mb', 'gpu_temperature_mean_celsius']:
        vals = [row.get(f"{metric}_{config}_Attempt{i}", '') for i in range(1,6)]
        vals_str = [f"{v:.4f}" if isinstance(v, float) else str(v) for v in vals]
        print(f"  {metric:35s} | {' | '.join(vals_str)}")

Instance: astropy__astropy-16058
Test: astropy/timeseries/periodograms/lombscargle/tests/test_lombscargle.py::test_nterms_methods[psd-4-full-True-False-fastchi2]
Repo: astropy/astropy

  BASE
  cpu_energy_joules                   | 46.0462 | 32.4057 | 31.4621 | 31.3817 | 31.3560
  gpu_energy_joules                   | 29.4947 | 22.1964 | 24.0243 | 22.1337 | 23.9968
  total_energy_joules                 | 75.5409 | 54.6021 | 55.4863 | 53.5155 | 55.3528
  system_energy_joules                | 158.0000 | nan | 244.0000 | 298.0000 | nan
  power_watts                         | 47.1031 | 45.4036 | 42.5581 | 44.4759 | 42.4573
  carbon_grams                        | 0.0052 | 0.0038 | 0.0039 | 0.0037 | 0.0038
  carbon_grams_system                 | 0.0110 | nan | 0.0169 | 0.0207 | nan
  duration_seconds                    | 1.6037 | 1.2026 | 1.3038 | 1.2032 | 1.3037
  cpu_usage_mean_percent              | 9.7429 | 12.0800 | 11.0455 | 12.3400 | 11.0727
  cpu_usage_peak_percent              | 36.

In [9]:
# Cella 5 - Colonne con valori vuoti
empty_counts = (df == '').sum() + df.isna().sum()
problematic = empty_counts[empty_counts > 0]
print(f"Colonne con celle vuote: {len(problematic)} su {len(df.columns)}\n")
for col, count in problematic.items():
    print(f"  {col}: {count} vuote su {len(df)}")

Colonne con celle vuote: 20 su 134

  system_energy_joules_base_Attempt1: 201 vuote su 878
  system_energy_joules_base_Attempt2: 410 vuote su 878
  system_energy_joules_base_Attempt3: 333 vuote su 878
  system_energy_joules_base_Attempt4: 263 vuote su 878
  system_energy_joules_base_Attempt5: 462 vuote su 878
  carbon_grams_system_base_Attempt1: 201 vuote su 878
  carbon_grams_system_base_Attempt2: 410 vuote su 878
  carbon_grams_system_base_Attempt3: 333 vuote su 878
  carbon_grams_system_base_Attempt4: 263 vuote su 878
  carbon_grams_system_base_Attempt5: 462 vuote su 878
  system_energy_joules_human_head_Attempt1: 215 vuote su 878
  system_energy_joules_human_head_Attempt2: 313 vuote su 878
  system_energy_joules_human_head_Attempt3: 445 vuote su 878
  system_energy_joules_human_head_Attempt4: 240 vuote su 878
  system_energy_joules_human_head_Attempt5: 462 vuote su 878
  carbon_grams_system_human_head_Attempt1: 215 vuote su 878
  carbon_grams_system_human_head_Attempt2: 313 vuote s